In [10]:
# 데이터 생성
import pandas as pd

dataset = pd.read_csv('../../data/csv/movie_reviews2.csv')
dataset.head()


,label,review
0,1,영화가 너무 재미있어서 하나도 안 지루해요.
1,0,영화가 하나도 안 재미있어서 너무 지루해요.
2,1,기대 안 했는데 생각보다 너무 괜찮네요.
3,0,기대 많이 했는데 생각보다 너무 별로네요.
4,1,배우 연기는 별로지만 스토리가 살렸습니다.


In [11]:
X = dataset.iloc[ : , -1]
y = dataset.iloc[ : , :-1]



In [12]:
# 형태소 분석
from konlpy.tag import Okt
okt = Okt()

def tokenized_korean(text_list):
  return [" ".join(okt.morphs(text, stem=True)) for text in text_list]

morphs_korean = tokenized_korean(X)

In [13]:
# 벡터화

from tensorflow.keras import layers, models

vectorize_layer = layers.TextVectorization(
  max_tokens = 1000,
  output_mode='int',
  output_sequence_length = 10	 # 단어 10개
)

vectorize_layer.adapt(morphs_korean)


In [14]:
# 파이프라인

import tensorflow as tf
train_ds = tf.data.Dataset.from_tensor_slices((morphs_korean, y)).batch(16)

In [15]:
# 모델 설계
def bulid_positional_encoding():
  
  # 입력층
  inputs = layers.Input((1,), dtype=tf.string) # 1은 입력받을 변수 (문장 하나씩 받으니까 1개)
  
	# 임베딩
  x = vectorize_layer(inputs)
  vocab_size = vectorize_layer.vocabulary_size()
  word_embedding = layers.Embedding(input_dim=vocab_size, output_dim=64)(x)
  
	# 포지셔널 인코딩
  # 위치 정보 추가 : 0~9 번호를 생성
  positions = tf.range(start=0, limit=10, delta=1)
  
	# 임베딩 : 위치를
  pos_embedding = layers.Embedding(input_dim=10, output_dim=64)(positions)
  
	# 의미 + 위치
  x = word_embedding + pos_embedding
  
	# 멀티 헤드 어텐션
  attention_output = layers.MultiHeadAttention(num_heads=2, key_dim=64)(x,x)
  
	# 잔차 계산 및 정규화
  x = layers.Add()([x, attention_output])
  x = layers.LayerNormalization()(x)
  
	# ffn
  ffn = layers.Dense(64, activation='relu')(x)
  
	# 잔차 연결 및 정규화
  x = layers.Add()([x, ffn])
  x = layers.LayerNormalization()(x)
  
	# 압축
  x = layers.GlobalAveragePooling1D()(x)
  
	# 출력층
  outputs = layers.Dense(1, activation='sigmoid')(x)
  return models.Model(inputs=inputs, outputs=outputs)

model = bulid_positional_encoding()
  

In [16]:
# 모델 설정
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [17]:
# 모델 학습
model.fit(train_ds, epochs=50)

Epoch 1/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - accuracy: 0.5000 - loss: 0.7850
Epoch 2/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.5000 - loss: 0.7041 
Epoch 3/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7000 - loss: 0.5771
Epoch 4/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.8000 - loss: 0.5203
Epoch 5/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8000 - loss: 0.4707 
Epoch 6/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9333 - loss: 0.4109 
Epoch 7/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9000 - loss: 0.3754 
Epoch 8/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9667 - loss: 0.3387
Epoch 9/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9333 - loss: 0.2959 
Epoch 10/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9667 - loss: 0.2659
Epoch 11/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9333 - loss: 0.2405 
Epoch 12/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9667 - loss: 0.2090
Epo

In [22]:
# 예측
# 테스트

test_text = ["영화가 너무 재미있어서 하나도 안 지루해요",
             "영화가 너무 지루해서 하나도 안 재미있어요",
             "인생 영화입니다"
						 ]

# 형태소별로 문장을 수정
test_morphs = tokenized_korean(test_text)
test_morphs

# 테스트 할 때 텐서 타입을 변환(문자열 이슈)
test_input = tf.constant(test_morphs)

# 예측 실행
predictions = model.predict(test_input)
predictions_size = len(predictions)
for i in range(predictions_size):
  p = predictions[i][0] * 100
  print(f'{test_text[i]} : {'긍정' if p > 50 else '부정'}({p:.2f}%)')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
영화가 너무 재미있어서 하나도 안 지루해요 : 긍정(78.89%)
영화가 너무 지루해서 하나도 안 재미있어요 : 긍정(74.49%)
인생 영화입니다 : 긍정(100.00%)
